In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import math
import os
import zipfile
import requests

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Data Loading and Preprocessing

First, we'll download the MovieLens 100k dataset, load it into a pandas DataFrame, and preprocess it for training by mapping user and item IDs to contiguous integers.

In [ ]:
# Define file paths and URLs
MOVIELENS_URL = 'http://files.grouplens.org/datasets/movielens/ml-100k.zip'
MOVIELENS_ZIP = 'ml-100k.zip'
MOVIELENS_DIR = 'ml-100k'
RATINGS_FILE = os.path.join(MOVIELENS_DIR, 'u.data')

# Download the dataset if not already present
if not os.path.exists(MOVIELENS_DIR):
    print(f"Downloading {MOVIELENS_URL}...")
    r = requests.get(MOVIELENS_URL)
    with open(MOVIELENS_ZIP, 'wb') as f:
        f.write(r.content)
    with zipfile.ZipFile(MOVIELENS_ZIP, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Download and extraction complete.")
else:
    print(f"Dataset already found at {MOVIELENS_DIR}.")

# Load the dataset
# The u.data file has no header, and columns are user ID, item ID, rating, timestamp
ratings_df = pd.read_csv(RATINGS_FILE, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

print(f"Total ratings: {len(ratings_df)}")

Dataset already found at ml-100k.
Total ratings: 100000


In [ ]:
# Map user and item IDs to contiguous integers
user_ids = ratings_df['user_id'].unique().tolist()
num_users = len(user_ids)
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
ratings_df['user_idx'] = ratings_df['user_id'].map(user_to_idx)

item_ids = ratings_df['item_id'].unique().tolist()
num_items = len(item_ids)
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}
ratings_df['item_idx'] = ratings_df['item_id'].map(item_to_idx)

print(f"Number of unique users: {num_users}")
print(f"Number of unique items: {num_items}")

ratings_df['binary_rating'] = ratings_df['rating'].apply(lambda x: 1 if x >= 4 else 0)


Number of unique users: 943
Number of unique items: 1682


In [ ]:
class BPRDataset(Dataset):
    def __init__(self, df, num_items, num_negatives=1):
        self.users = df['user_idx'].values
        self.positive_items = df['item_idx'].values
        self.num_items = num_items
        self.num_negatives = num_negatives

        # Create a set of (user, item) pairs for efficient lookup during negative sampling
        self.user_item_interactions = df.groupby('user_idx')['item_idx'].apply(set).to_dict()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user_idx = self.users[idx]
        positive_item_idx = self.positive_items[idx]

        # Sample negative items
        negative_item_idx = self._sample_negative(user_idx)

        return (
            torch.tensor(user_idx, dtype=torch.long),
            torch.tensor(positive_item_idx, dtype=torch.long),
            torch.tensor(negative_item_idx, dtype=torch.long)
        )

    def _sample_negative(self, user_idx):
        user_positive_items = self.user_item_interactions.get(user_idx, set())
        while True:
            negative_item = np.random.randint(0, self.num_items)
            if negative_item not in user_positive_items:
                return negative_item

num_negatives_sample = 1
batch_size = 1024

# Sort ratings by timestamp for chronological split
ratings_df = ratings_df.sort_values(by='timestamp').reset_index(drop=True)

# Determine split point for 80% train, 20% test
split_idx = int(len(ratings_df) * 0.8)
train_df = ratings_df.iloc[:split_idx]
test_df = ratings_df.iloc[split_idx:]

# train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)

def balance_test_set_by_item_popularity(df, max_interactions_per_item=20):
    balanced_df_rows = []
    # Group by item_idx and sample if count exceeds max_interactions_per_item
    for item_id, group in df.groupby('item_idx'):
        if len(group) > max_interactions_per_item:
            # Randomly sample 'max_interactions_per_item' rows from the group
            balanced_df_rows.append(group.sample(n=max_interactions_per_item, random_state=42)) # Using a fixed random_state for reproducibility
        else:
            balanced_df_rows.append(group)

    # Concatenate all sampled/kept rows to form the new balanced test_df
    if balanced_df_rows:
        return pd.concat(balanced_df_rows).sort_values(by='timestamp').reset_index(drop=True)
    else:
        return pd.DataFrame(columns=df.columns) # Return empty DataFrame with original columns if no data

print(f"Original test_df size: {len(test_df)}")
test_df = balance_test_set_by_item_popularity(test_df, max_interactions_per_item=20)
print(f"Balanced test_df size: {len(test_df)}")

# Instantiate BPR Datasets and DataLoaders
train_bpr_dataset = BPRDataset(train_df, num_items, num_negatives=num_negatives_sample)
test_bpr_dataset = BPRDataset(test_df, num_items, num_negatives=num_negatives_sample)

train_bpr_loader = DataLoader(train_bpr_dataset, batch_size=batch_size, shuffle=True)
test_bpr_loader = DataLoader(test_bpr_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of batches in BPR training loader: {len(train_bpr_loader)}")
print(f"Number of batches in BPR test loader: {len(test_bpr_loader)}")

Original test_df size: 20000
Balanced test_df size: 13643
Number of batches in BPR training loader: 79
Number of batches in BPR test loader: 14


We need to consturct a sparse interaction matrix for LightGCN.

In [ ]:
import scipy.sparse as sp

# Construct the adjacency matrix (user-item interactions) for LightGCN
# This is typically an (M+N) x (M+N) matrix where M=users, N=items
# The blocks are [[0, R], [R^T, 0]], where R is the user-item interaction matrix.
R_sparse = sp.csr_matrix((np.ones_like(train_df['user_idx'], dtype=np.float32),
                         (train_df['user_idx'], train_df['item_idx'])),
                        shape=(num_users, num_items))

# Adjacency matrix A
adj = sp.bmat([[sp.csr_matrix((num_users, num_users)), R_sparse],
               [R_sparse.transpose(), sp.csr_matrix((num_items, num_items))]],
              format='csr')

# Degree matrix D and its inverse square root
rowsum = np.array(adj.sum(axis=1))
d_inv_sqrt = np.power(rowsum, -0.5).flatten()
d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
D_inv_sqrt = sp.diags(d_inv_sqrt)

# Normalized adjacency matrix A_hat = D^{-1/2} A D^{-1/2}
adj_normalized = D_inv_sqrt.dot(adj).dot(D_inv_sqrt).tocoo()

# Convert to PyTorch sparse tensor and move to device
row = torch.tensor(adj_normalized.row, dtype=torch.long)
col = torch.tensor(adj_normalized.col, dtype=torch.long)
index = torch.stack([row, col])
data = torch.tensor(adj_normalized.data, dtype=torch.float)
adj_normalized_torch = torch.sparse_coo_tensor(index, data, adj_normalized.shape).to(device)



/tmp/ipykernel_2032/3944916455.py:17: RuntimeWarning: divide by zero encountered in power
  d_inv_sqrt = np.power(rowsum, -0.5).flatten()


In [ ]:


# Calculate item popularity (phi(i)) from the training set
item_popularity_counts = train_df['item_idx'].value_counts().to_dict()

# Identify long-tail items (bottom 80% by review count)
all_item_indices = np.arange(num_items)
item_counts_series = pd.Series(item_popularity_counts).reindex(all_item_indices, fill_value=0)

# Sort items by their review count
sorted_items_by_popularity = item_counts_series.sort_values(ascending=True)

# Define long tail as the 80% of items with the least reviews
long_tail_threshold_idx = int(len(sorted_items_by_popularity) * 0.8)
long_tail_item_indices = set(sorted_items_by_popularity.index[:long_tail_threshold_idx].tolist())

print(f"Number of long-tail items: {len(long_tail_item_indices)}")


Number of long-tail items: 1345


## Base Models

We implement Matrix Factorization (MF) and LightGCN

In [ ]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim):
        super(MatrixFactorization, self).__init__()
        self.user_embeddings = nn.Embedding(num_users, embedding_dim)
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)

        # Initialize embeddings with a small normal distribution
        nn.init.normal_(self.user_embeddings.weight, std=0.01)
        nn.init.normal_(self.item_embeddings.weight, std=0.01)

    def forward(self, user_indices, item_indices):
        user_embedding = self.user_embeddings(user_indices)
        item_embedding = self.item_embeddings(item_indices)
        return (user_embedding * item_embedding).sum(dim=1)


In [ ]:
class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, num_layers, adj_normalized_torch):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        self.adj_normalized_torch = adj_normalized_torch

        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)

        # Initialize embeddings with a small normal distribution
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, user_indices, item_indices):
        # Initial embeddings E0 = [User_Embeddings; Item_Embeddings]
        all_embeddings = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        embeddings_list = [all_embeddings]

        for layer in range(self.num_layers):
            # E_k+1 = A_hat E_k
            all_embeddings = torch.sparse.mm(self.adj_normalized_torch, all_embeddings)
            embeddings_list.append(all_embeddings)

        # GCN aggregation: mean of embeddings from all layers (original LightGCN)
        final_all_embeddings = torch.mean(torch.stack(embeddings_list), dim=0)

        # Split final user and item embeddings
        final_user_embeddings, final_item_embeddings = torch.split(
            final_all_embeddings, [self.num_users, self.num_items]
        )

        # Retrieve embeddings for the given user_indices and item_indices
        user_emb = final_user_embeddings[user_indices]
        item_emb = final_item_embeddings[item_indices]

        # Dot product for prediction
        prediction = torch.sum(user_emb * item_emb, dim=1)
        return prediction

## Metrics


In [ ]:
def calculate_ndcg_at_k(model, test_df, train_user_item_interactions, num_items, k=20):
    model.eval()
    ndcgs = []
    # Get unique users from the test set that have at least one positive rating
    test_users = test_df[test_df['binary_rating'] == 1]['user_idx'].unique()

    with torch.no_grad():
        for user_idx in test_users:
            # Ground truth: items user rated positively in the test set
            true_pos_items = set(test_df[(test_df['user_idx'] == user_idx) & (test_df['binary_rating'] == 1)]['item_idx'].unique())
            if not true_pos_items:
                continue

            # Items user interacted with in training (to exclude from recommendations)
            seen_items = train_user_item_interactions.get(user_idx, set())

            # Prepare all candidate items (excluding seen items)
            candidate_items_list = [item for item in range(num_items) if item not in seen_items]
            if not candidate_items_list:
                continue

            # Predict scores for all candidate items for the current user
            user_tensor = torch.tensor([user_idx] * len(candidate_items_list), dtype=torch.long).to(device)
            candidate_items_tensor = torch.tensor(candidate_items_list, dtype=torch.long).to(device)
            scores = model(user_tensor, candidate_items_tensor)

            # Rank items by score (descending)
            ranked_item_indices = torch.argsort(scores, descending=True).cpu().numpy()
            ranked_items_pred = [candidate_items_list[i] for i in ranked_item_indices]

            # Calculate DCG
            dcg = 0.0
            for rank_idx, item_id in enumerate(ranked_items_pred[:k]):
                if item_id in true_pos_items:
                    dcg += 1.0 / math.log2(rank_idx + 2) # rank_idx is 0-indexed, so rank is rank_idx + 1

            # Calculate IDCG
            idcg = 0.0
            for rank_idx in range(min(k, len(true_pos_items))):
                idcg += 1.0 / math.log2(rank_idx + 2)

            if idcg > 0:
                ndcgs.append(dcg / idcg)
            else:
                ndcgs.append(0.0)

    return np.mean(ndcgs) if ndcgs else 0.0

In [ ]:
def calculate_arp(model, test_df, item_popularity_counts, num_items, k=20):
    model.eval()
    arps = []
    test_users = test_df['user_idx'].unique()
    train_user_item_interactions = train_bpr_dataset.user_item_interactions

    with torch.no_grad():
        for user_idx in test_users:
            seen_items = train_user_item_interactions.get(user_idx, set())
            candidate_items_list = [item for item in range(num_items) if item not in seen_items]
            if not candidate_items_list:
                continue

            user_tensor = torch.tensor([user_idx] * len(candidate_items_list), dtype=torch.long).to(device)
            candidate_items_tensor = torch.tensor(candidate_items_list, dtype=torch.long).to(device)
            scores = model(user_tensor, candidate_items_tensor)

            ranked_item_indices = torch.argsort(scores, descending=True).cpu().numpy()
            ranked_items_pred = [candidate_items_list[i] for i in ranked_item_indices]

            total_popularity = 0
            count_recommended = 0
            for item_id in ranked_items_pred[:k]:
                total_popularity += item_popularity_counts.get(item_id, 0)
                count_recommended += 1

            if count_recommended > 0:
                arps.append(total_popularity / count_recommended)
            else:
                arps.append(0.0)

    return np.mean(arps) if arps else 0.0

def calculate_aplt(model, test_df, long_tail_item_indices, num_items, k=20):
    model.eval()
    aplts = []
    test_users = test_df['user_idx'].unique()
    train_user_item_interactions = train_bpr_dataset.user_item_interactions

    with torch.no_grad():
        for user_idx in test_users:
            seen_items = train_user_item_interactions.get(user_idx, set())
            candidate_items_list = [item for item in range(num_items) if item not in seen_items]
            if not candidate_items_list:
                continue

            user_tensor = torch.tensor([user_idx] * len(candidate_items_list), dtype=torch.long).to(device)
            candidate_items_tensor = torch.tensor(candidate_items_list, dtype=torch.long).to(device)
            scores = model(user_tensor, candidate_items_tensor)

            ranked_item_indices = torch.argsort(scores, descending=True).cpu().numpy()
            ranked_items_pred = [candidate_items_list[i] for i in ranked_item_indices]

            long_tail_in_rec = 0
            count_recommended = 0
            for item_id in ranked_items_pred[:k]:
                if item_id in long_tail_item_indices:
                    long_tail_in_rec += 1
                count_recommended += 1

            if count_recommended > 0:
                aplts.append(long_tail_in_rec / count_recommended)
            else:
                aplts.append(0.0)

    return np.mean(aplts) if aplts else 0.0

def calculate_aclt(model, test_df, long_tail_item_indices, num_items, k=20):
    model.eval()
    unique_long_tail_recommended = set()
    test_users = test_df['user_idx'].unique()
    train_user_item_interactions = train_bpr_dataset.user_item_interactions

    with torch.no_grad():
        for user_idx in test_users:
            seen_items = train_user_item_interactions.get(user_idx, set())
            candidate_items_list = [item for item in range(num_items) if item not in seen_items]
            if not candidate_items_list:
                continue

            user_tensor = torch.tensor([user_idx] * len(candidate_items_list), dtype=torch.long).to(device)
            candidate_items_tensor = torch.tensor(candidate_items_list, dtype=torch.long).to(device)
            scores = model(user_tensor, candidate_items_tensor)

            ranked_item_indices = torch.argsort(scores, descending=True).cpu().numpy()
            ranked_items_pred = [candidate_items_list[i] for i in ranked_item_indices]

            for item_id in ranked_items_pred[:k]:
                if item_id in long_tail_item_indices:
                    unique_long_tail_recommended.add(item_id)

    if len(long_tail_item_indices) > 0:
        return len(unique_long_tail_recommended) / len(long_tail_item_indices)
    else:
        return 0.0

In [ ]:
import numpy as np
from scipy.spatial.distance import jensenshannon

# Helper function to categorize items into popularity groups for CP reranking
def build_popularity_groups_for_cp(item_counts_series, num_items):
    # Sort items by their review count (popularity)
    sorted_items = item_counts_series.sort_values(ascending=True)

    # Define groups based on quantiles (e.g., 20% head, 60% mid, 20% tail, as often used in CP)
    n_items = len(sorted_items)
    head_threshold = int(0.8 * n_items) # Most popular 20%
    tail_threshold = int(0.2 * n_items) # Least popular 20%

    # Map items to 0 (head), 1 (mid), 2 (tail)
    item_to_group = {}
    for i, item_idx in enumerate(sorted_items.index):
        if i < tail_threshold: # Least popular 20%
            item_to_group[item_idx] = 2 # Tail
        elif i >= head_threshold: # Most popular 20%
            item_to_group[item_idx] = 0 # Head
        else: # Middle 60%
            item_to_group[item_idx] = 1 # Mid

    def get_group(item_idx):
        return item_to_group.get(item_idx, 1) # Default to mid if not found (shouldn't happen with all_item_indices)

    return get_group


# Define the get_group_func using the item_counts_series
get_group_func = build_popularity_groups_for_cp(item_counts_series, num_items)

# Create the global item_to_group mapping needed for the UPD metrics
item_to_group = {i: get_group_func(i) for i in range(num_items)}

print(f"Successfully created item_to_group mapping for {len(item_to_group)} items.")

def calculate_propensity(items, item_to_group, num_groups, weights=None, epsilon=1e-9):
    counts = np.zeros(num_groups) + epsilon # Add smoothing to avoid division by zero or log(0)
    if weights is None:
        weights = [1.0] * len(items)
    for item, weight in zip(items, weights):
        group_idx = item_to_group.get(item)
        if group_idx is not None:
            counts[group_idx] += weight
    total = np.sum(counts)
    return counts / total if total > 0 else counts / num_groups

def calculate_upd_at_k(model, test_df, train_df, item_to_group, num_items, k=20, num_groups=3):
    model.eval()
    upd_scores = []
    test_users = test_df['user_idx'].unique()
    train_user_interactions = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
    train_user_ratings = train_df.groupby('user_idx')['rating'].apply(list).to_dict()

    with torch.no_grad():
        for user_idx in test_users:
            user_history_items = train_user_interactions.get(user_idx, [])
            user_history_ratings = train_user_ratings.get(user_idx, [])
            if not user_history_items:
                continue

            seen_items = set(user_history_items)
            candidate_items = [i for i in range(num_items) if i not in seen_items]
            if not candidate_items:
                continue

            user_tensor = torch.tensor([user_idx] * len(candidate_items), dtype=torch.long).to(device)
            item_tensor = torch.tensor(candidate_items, dtype=torch.long).to(device)
            scores = model(user_tensor, item_tensor)

            top_k_indices = torch.argsort(scores, descending=True)[:k].cpu().numpy()
            rec_list = [candidate_items[i] for i in top_k_indices]

            P = calculate_propensity(user_history_items, item_to_group, num_groups, user_history_ratings)
            Q = calculate_propensity(rec_list, item_to_group, num_groups)

            js_dist = jensenshannon(P, Q)
            upd_scores.append(js_dist**2)

    return np.mean(upd_scores) if upd_scores else 0.0

Successfully created item_to_group mapping for 1682 items.


## Training Base Models

In [ ]:
from tqdm.auto import tqdm

class BPRLoss(nn.Module):
    def forward(self, pos_pred, neg_pred):
        return -torch.sum(nn.functional.logsigmoid(pos_pred - neg_pred))

compute_bpr_loss = BPRLoss()

# Pre-calculate item groups for UPD metric
get_group_func = build_popularity_groups_for_cp(item_counts_series, num_items)
item_to_group = {i: get_group_func(i) for i in range(num_items)}

def train(model, lr=1e-3, epochs=20):
    model = model.to(device)
    criterion = BPRLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in tqdm(range(epochs)):
        model.train()
        total_loss = 0
        for user, pos_item, neg_item in train_bpr_loader:
            user, pos_item, neg_item = user.to(device), pos_item.to(device), neg_item.to(device)
            optimizer.zero_grad()
            pos_pred = model(user, pos_item)
            neg_pred = model(user, neg_item)
            loss = criterion(pos_pred, neg_pred)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_bpr_loader)
        model.eval()

        if (epoch + 1) % 5 == 0:
          ndcg_5 = calculate_ndcg_at_k(model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=5)
          ndcg_20 = calculate_ndcg_at_k(model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=20)
          arp = calculate_arp(model, test_df, item_popularity_counts, num_items, k=20)
          aplt = calculate_aplt(model, test_df, long_tail_item_indices, num_items, k=20)
          upd = calculate_upd_at_k(model, test_df, train_df, item_to_group, num_items, k=20)

          print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_train_loss:.4f}, NDCG@20: {ndcg_20:.4f}, ARP: {arp:.4f}, APLT: {aplt:.4f}, UPD: {upd:.4f}")
        else:
          ndcg_5 = calculate_ndcg_at_k(model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=5)
          print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_train_loss:.4f}, NDCG@5: {ndcg_5:.4f}")

In [ ]:
embedding_dim = 50

mf_model = MatrixFactorization(num_users, num_items, embedding_dim).to(device)
train(mf_model, epochs=50)

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/50, Loss: 699.6183, NDCG@5: 0.0246
Epoch 2/50, Loss: 601.7616, NDCG@5: 0.0290
Epoch 3/50, Loss: 412.3773, NDCG@5: 0.0359
Epoch 4/50, Loss: 342.9254, NDCG@5: 0.0337
Epoch 5/50, Loss: 319.8983, NDCG@20: 0.0405, ARP: 192.2836, APLT: 0.3109, UPD: 0.1292
Epoch 6/50, Loss: 305.4596, NDCG@5: 0.0306
Epoch 7/50, Loss: 297.0290, NDCG@5: 0.0313
Epoch 8/50, Loss: 285.0968, NDCG@5: 0.0337
Epoch 9/50, Loss: 275.5412, NDCG@5: 0.0347
Epoch 10/50, Loss: 267.0706, NDCG@20: 0.0446, ARP: 181.4464, APLT: 0.3203, UPD: 0.1259
Epoch 11/50, Loss: 257.7376, NDCG@5: 0.0335
Epoch 12/50, Loss: 249.2844, NDCG@5: 0.0303
Epoch 13/50, Loss: 243.8008, NDCG@5: 0.0318
Epoch 14/50, Loss: 237.8741, NDCG@5: 0.0329
Epoch 15/50, Loss: 229.6135, NDCG@20: 0.0426, ARP: 170.0693, APLT: 0.3353, UPD: 0.1187
Epoch 16/50, Loss: 227.6182, NDCG@5: 0.0335
Epoch 17/50, Loss: 222.1934, NDCG@5: 0.0347
Epoch 18/50, Loss: 218.6123, NDCG@5: 0.0360
Epoch 19/50, Loss: 215.3535, NDCG@5: 0.0342
Epoch 20/50, Loss: 213.2594, NDCG@20: 0.0421

# MACR Models and Training

In [ ]:
class MACRDataset(Dataset):
    def __init__(self, df):
        self.users = df['user_idx'].values
        self.items = df['item_idx'].values
        self.labels = df['binary_rating'].values.astype(np.float32)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.users[idx], dtype=torch.long),
            torch.tensor(self.items[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float)
        )

# Instantiate MACR Datasets and DataLoaders
macr_train_dataset = MACRDataset(train_df)
macr_test_dataset = MACRDataset(test_df)

macr_train_loader = DataLoader(macr_train_dataset, batch_size=batch_size, shuffle=True)
macr_test_loader = DataLoader(macr_test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm

class MACRWrapper(nn.Module):
    """
    Wraps any base recommender model with the MACR (Model-Agnostic Counterfactual Reasoning)
    framework to eliminate popularity bias.
    """
    def __init__(self, base_model, num_users, num_items, alpha=1e-3, beta=1e-3, c=30.0):
        super(MACRWrapper, self).__init__()
        self.base_model = base_model

        # MACR Specific Branches: 1D embeddings to capture popularity/activity bias
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)

        # Hyperparameters
        self.alpha = alpha  # weight for item popularity loss
        self.beta = beta    # weight for user activity loss
        self.c = c          # counterfactual reference constant

        # Initialize bias weights to zero
        nn.init.constant_(self.user_bias.weight, 0.0)
        nn.init.constant_(self.item_bias.weight, 0.0)

    def forward(self, user_indices, item_indices):
        # 1. Get personalized matching score from base model
        y_ui = self.base_model(user_indices, item_indices)
        if len(y_ui.shape) == 1:
            y_ui = y_ui.unsqueeze(-1)

        # 2. Compute Popularity Branches
        y_i = self.item_bias(item_indices)
        y_u = self.user_bias(user_indices)

        if self.training:
            return y_ui, y_i, y_u
        else:
            # Perform Counterfactual Inference (TIE = TE - NDE)
            sig_ui = torch.sigmoid(y_ui)
            sig_i = torch.sigmoid(y_i)
            sig_u = torch.sigmoid(y_u)

            c_val = torch.tensor(self.c).to(y_ui.device)
            score = (sig_ui - torch.sigmoid(c_val)) * sig_i * sig_u
            return score.squeeze()

    def compute_bce_loss(self, user_indices, item_indices, labels):
        y_ui, y_i, y_u = self.forward(user_indices, item_indices)
        y_total = torch.sigmoid(y_ui) * torch.sigmoid(y_i) * torch.sigmoid(y_u)

        loss_main = F.binary_cross_entropy(y_total.squeeze(), labels)
        loss_item = F.binary_cross_entropy(torch.sigmoid(y_i).squeeze(), labels)
        loss_user = F.binary_cross_entropy(torch.sigmoid(y_u).squeeze(), labels)

        return loss_main + self.alpha * loss_item + self.beta * loss_user

    def compute_bpr_loss(self, user_indices, pos_item_indices, neg_item_indices):
        # Calculate scores for positive samples
        y_ui_pos, y_i_pos, y_u_pos = self.forward(user_indices, pos_item_indices)
        y_total_pos = torch.sigmoid(y_ui_pos) * torch.sigmoid(y_i_pos) * torch.sigmoid(y_u_pos)

        # Calculate scores for negative samples
        y_ui_neg, y_i_neg, _ = self.forward(user_indices, neg_item_indices)
        y_total_neg = torch.sigmoid(y_ui_neg) * torch.sigmoid(y_i_neg) * torch.sigmoid(y_u_pos)

        # Standard BPR loss on combined probability
        loss_main_bpr = compute_bpr_loss(y_total_pos.squeeze(), y_total_neg.squeeze())

        # Item popularity loss for both positive and negative items
        pos_labels = torch.ones_like(user_indices, dtype=torch.float).to(device)
        neg_labels = torch.zeros_like(pos_labels)
        loss_item = F.binary_cross_entropy(torch.sigmoid(y_i_pos).squeeze(), pos_labels) + \
                    F.binary_cross_entropy(torch.sigmoid(y_i_neg).squeeze(), neg_labels)

        # User activity loss (calculated once for the user branch)
        loss_user = F.binary_cross_entropy(torch.sigmoid(y_u_pos).squeeze(), pos_labels)

        return loss_main_bpr + self.alpha * loss_item + self.beta * loss_user

In [ ]:
def train_macr_bce(macr_model, lr=1e-3, epochs=20):
    macr_model = macr_model.to(device)
    optimizer = optim.Adam(macr_model.parameters(), lr=lr)

    for epoch in tqdm(range(epochs), desc="MACR BCE Training"):
        macr_model.train()
        total_loss = 0
        for user, item, label in macr_train_loader:
            user, item, label = user.to(device), item.to(device), label.to(device)
            optimizer.zero_grad()
            loss = macr_model.compute_bce_loss(user, item, label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_bpr_loader)

        # Evaluation Phase
        macr_model.eval()

        if (epoch + 1) % 5 == 0:
          ndcg_5 = calculate_ndcg_at_k(macr_model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=5)
          ndcg_20 = calculate_ndcg_at_k(macr_model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=20)
          arp = calculate_arp(macr_model, test_df, item_popularity_counts, num_items, k=20)
          aplt = calculate_aplt(macr_model, test_df, long_tail_item_indices, num_items, k=20)
          aclt = calculate_aclt(macr_model, test_df, long_tail_item_indices, num_items, k=20)
          upd = calculate_upd_at_k(macr_model, test_df, train_df, item_to_group, num_items, k=20)


          print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Test NDCG@5: {ndcg_5:.4f}, Test NDCG@20: {ndcg_20:.4f}, ARP: {arp:.4f}, APLT: {aplt:.4f}, ACLT: {aclt:.4f}, UPD: {upd:.4f}")

        else:
          ndcg_5 = calculate_ndcg_at_k(macr_model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=5)
          print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Test NDCG@5: {ndcg_5:.4f}")



In [ ]:
def train_macr_bpr(macr_model, lr=1e-3, epochs=20):
    macr_model = macr_model.to(device)
    optimizer = optim.Adam(macr_model.parameters(), lr=lr)

    for epoch in tqdm(range(epochs), desc="MACR BPR Training"):
        macr_model.train()
        total_loss = 0
        for user, pos_item, neg_item in train_bpr_loader:
            user, pos_item, neg_item = user.to(device), pos_item.to(device), neg_item.to(device)

            optimizer.zero_grad()

            # Create labels for positive items (all 1s for BPR context)
            loss = macr_model.compute_bpr_loss(user, pos_item, neg_item)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_bpr_loader)

        # Evaluation Phase
        macr_model.eval()

        if (epoch + 1) % 5 == 0:
          ndcg_5 = calculate_ndcg_at_k(macr_model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=5)
          ndcg_20 = calculate_ndcg_at_k(macr_model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=20)
          arp = calculate_arp(macr_model, test_df, item_popularity_counts, num_items, k=20)
          aplt = calculate_aplt(macr_model, test_df, long_tail_item_indices, num_items, k=20)
          aclt = calculate_aclt(macr_model, test_df, long_tail_item_indices, num_items, k=20)
          upd = calculate_upd_at_k(macr_model, test_df, train_df, item_to_group, num_items, k=20)


          print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Test NDCG@5: {ndcg_5:.4f}, Test NDCG@20: {ndcg_20:.4f}, ARP: {arp:.4f}, APLT: {aplt:.4f}, ACLT: {aclt:.4f}, UPD: {upd:.4f}")

        else:
          ndcg_5 = calculate_ndcg_at_k(macr_model, test_df, train_bpr_dataset.user_item_interactions, num_items, k=5)
          print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Test NDCG@5: {ndcg_5:.4f}")


In [ ]:
# Instantiate MACRWrapper with the trained MF model as the base model
mf_for_macr = MatrixFactorization(num_users, num_items, embedding_dim).to(device)
macr_mf_model = MACRWrapper(mf_for_macr, num_users, num_items).to(device)
train_macr_bpr(macr_mf_model, epochs=30)

MACR BPR Training:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1/30, Train Loss: 700.9303, Test NDCG@5: 0.0052
Epoch 2/30, Train Loss: 690.4808, Test NDCG@5: 0.0152
Epoch 3/30, Train Loss: 659.2693, Test NDCG@5: 0.0145
Epoch 4/30, Train Loss: 634.7567, Test NDCG@5: 0.0160
Epoch 5/30, Train Loss: 622.6239, Test NDCG@5: 0.0135, Test NDCG@20: 0.0191, ARP: 96.2705, APLT: 0.6587, ACLT: 0.0379, UPD: 0.1333
Epoch 6/30, Train Loss: 614.9672, Test NDCG@5: 0.0132
Epoch 7/30, Train Loss: 608.3381, Test NDCG@5: 0.0107
Epoch 8/30, Train Loss: 603.6542, Test NDCG@5: 0.0120
Epoch 9/30, Train Loss: 599.1156, Test NDCG@5: 0.0125
Epoch 10/30, Train Loss: 594.2462, Test NDCG@5: 0.0142, Test NDCG@20: 0.0190, ARP: 91.7321, APLT: 0.6587, ACLT: 0.0409, UPD: 0.1333
Epoch 11/30, Train Loss: 590.2873, Test NDCG@5: 0.0118
Epoch 12/30, Train Loss: 585.8613, Test NDCG@5: 0.0117
Epoch 13/30, Train Loss: 581.9842, Test NDCG@5: 0.0122
Epoch 14/30, Train Loss: 578.1340, Test NDCG@5: 0.0131
Epoch 15/30, Train Loss: 573.8679, Test NDCG@5: 0.0122, Test NDCG@20: 0.0202, ARP: 87

In [ ]:
mf_for_macr_2 = MatrixFactorization(num_users, num_items, embedding_dim).to(device)
macr_mf_model_2 = MACRWrapper(mf_for_macr_2, num_users, num_items).to(device)
train_macr_bce(macr_mf_model_2, epochs=25)

MACR BCE Training:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/25, Train Loss: 1.1951, Test NDCG@5: 0.0010
Epoch 2/25, Train Loss: 1.1215, Test NDCG@5: 0.0113
Epoch 3/25, Train Loss: 0.9814, Test NDCG@5: 0.0147
Epoch 4/25, Train Loss: 0.8847, Test NDCG@5: 0.0124
Epoch 5/25, Train Loss: 0.8387, Test NDCG@5: 0.0142, Test NDCG@20: 0.0181, ARP: 92.9449, APLT: 0.6625, ACLT: 0.0216, UPD: 0.1254
Epoch 6/25, Train Loss: 0.8103, Test NDCG@5: 0.0143
Epoch 7/25, Train Loss: 0.7901, Test NDCG@5: 0.0152
Epoch 8/25, Train Loss: 0.7718, Test NDCG@5: 0.0160
Epoch 9/25, Train Loss: 0.7574, Test NDCG@5: 0.0166
Epoch 10/25, Train Loss: 0.7448, Test NDCG@5: 0.0165, Test NDCG@20: 0.0223, ARP: 87.4775, APLT: 0.6674, ACLT: 0.0506, UPD: 0.1293
Epoch 11/25, Train Loss: 0.7324, Test NDCG@5: 0.0166
Epoch 12/25, Train Loss: 0.7208, Test NDCG@5: 0.0179
Epoch 13/25, Train Loss: 0.7108, Test NDCG@5: 0.0182
Epoch 14/25, Train Loss: 0.7003, Test NDCG@5: 0.0184
Epoch 15/25, Train Loss: 0.6919, Test NDCG@5: 0.0184, Test NDCG@20: 0.0238, ARP: 82.0247, APLT: 0.6701, ACLT: 0.0

In [ ]:
torch.save(mf_model, 'mf.pth')

In [ ]:
for param in mf_model.parameters():
    param.requires_grad = False

pretrained_macr_mf_model = MACRWrapper(mf_model, num_users, num_items, c=20).to(device)
train_macr_bce(pretrained_macr_mf_model, epochs=25)

MACR BCE Training:   0%|          | 0/25 [00:00<?, ?it/s]

Epoch 1/25, Train Loss: 0.9199, Test NDCG@5: 0.0218
Epoch 2/25, Train Loss: 0.8992, Test NDCG@5: 0.0186
Epoch 3/25, Train Loss: 0.8810, Test NDCG@5: 0.0175
Epoch 4/25, Train Loss: 0.8621, Test NDCG@5: 0.0166
Epoch 5/25, Train Loss: 0.8466, Test NDCG@5: 0.0164, Test NDCG@20: 0.0266, ARP: 77.5829, APLT: 0.6863, ACLT: 0.1680, UPD: 0.0702
Epoch 6/25, Train Loss: 0.8301, Test NDCG@5: 0.0169
Epoch 7/25, Train Loss: 0.8149, Test NDCG@5: 0.0170
Epoch 8/25, Train Loss: 0.8013, Test NDCG@5: 0.0170
Epoch 9/25, Train Loss: 0.7886, Test NDCG@5: 0.0170
Epoch 10/25, Train Loss: 0.7765, Test NDCG@5: 0.0174, Test NDCG@20: 0.0257, ARP: 76.3490, APLT: 0.6954, ACLT: 0.1524, UPD: 0.0677
Epoch 11/25, Train Loss: 0.7653, Test NDCG@5: 0.0174
Epoch 12/25, Train Loss: 0.7545, Test NDCG@5: 0.0174
Epoch 13/25, Train Loss: 0.7441, Test NDCG@5: 0.0175
Epoch 14/25, Train Loss: 0.7349, Test NDCG@5: 0.0174
Epoch 15/25, Train Loss: 0.7255, Test NDCG@5: 0.0174, Test NDCG@20: 0.0257, ARP: 75.7918, APLT: 0.6966, ACLT: 0.1

## xQuad and CP

In [ ]:
import pandas as pd
import numpy as np
import torch
import math

# Derive short_head_item_indices from already computed long_tail_item_indices
all_item_indices_set = set(range(num_items))
short_head_item_indices = all_item_indices_set - long_tail_item_indices

def smooth_xquad_rerank(initial_list, user_history_items, lambda_val=0.5, top_k=10):
    if not initial_list:
        return []

    scores_only = [score for _, score in initial_list]
    max_score_in_list = max(scores_only)
    min_score_in_list = min(scores_only)

    norm_predictions = {}
    if max_score_in_list == min_score_in_list:
        norm_predictions = {iid: 0.5 for iid, _ in initial_list}
    else:
        norm_predictions = {
            iid: (score - min_score_in_list) / (max_score_in_list - min_score_in_list)
            for iid, score in initial_list
        }

    user_lt_count = sum(1 for iid in user_history_items if iid in long_tail_item_indices)
    total_user_ratings = len(user_history_items)

    p_long_tail_u = user_lt_count / total_user_ratings if total_user_ratings > 0 else 0.5
    p_short_head_u = 1.0 - p_long_tail_u

    S = []
    R_items = [iid for iid, _ in initial_list]

    while len(S) < top_k and R_items:
        best_item = None
        max_score = -float('inf')

        lt_in_S_count = sum(1 for i in S if i in long_tail_item_indices)
        sh_in_S_count = sum(1 for i in S if i in short_head_item_indices)

        lt_ratio_in_S = lt_in_S_count / len(S) if len(S) > 0 else 0.0
        sh_ratio_in_S = sh_in_S_count / len(S) if len(S) > 0 else 0.0

        for iid in R_items:
            p_v_u = norm_predictions[iid]

            diversity_bonus = 0.0
            if iid in long_tail_item_indices:
                diversity_bonus = p_long_tail_u * (1 - lt_ratio_in_S)
            elif iid in short_head_item_indices:
                diversity_bonus = p_short_head_u * (1 - sh_ratio_in_S)

            score = (1 - lambda_val) * p_v_u + lambda_val * diversity_bonus

            if score > max_score:
                max_score = score
                best_item = iid

        if best_item is not None:
            S.append(best_item)
            R_items.remove(best_item)
        else:
            break

    return S

In [ ]:
def evaluate_recommendation_list(user_idx, recommended_items_list, k, test_df, item_popularity_counts, long_tail_item_indices):
    true_pos_items = set(test_df[(test_df['user_idx'] == user_idx) & (test_df['binary_rating'] == 1)]['item_idx'].unique())
    dcg = 0.0
    for rank_idx, item_id in enumerate(recommended_items_list[:k]):
        if item_id in true_pos_items: dcg += 1.0 / math.log2(rank_idx + 2)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(k, len(true_pos_items))))
    ndcg_k = dcg / idcg if idcg > 0 else 0.0
    arp_k = np.mean([item_popularity_counts.get(iid, 0) for iid in recommended_items_list[:k]])
    aplt_k = np.mean([1 if iid in long_tail_item_indices else 0 for iid in recommended_items_list[:k]])
    return ndcg_k, arp_k, aplt_k

def evaluate_xquad_reranking_on_test_set(model, test_df, train_user_item_interactions, num_items, item_popularity_counts, long_tail_item_indices, k=10, lambda_val=0.85):
    model.eval()
    train_user_interactions = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
    train_user_ratings = train_df.groupby('user_idx')['rating'].apply(list).to_dict()

    base_metrics = defaultdict(list)
    reranked_metrics = defaultdict(list)

    with torch.no_grad():
        for user_idx in test_df['user_idx'].unique():
            user_history_items = train_user_item_interactions.get(user_idx, set())
            all_candidates = [i for i in range(num_items) if i not in user_history_items]
            if not all_candidates: continue
            u_t = torch.tensor([user_idx]*len(all_candidates)).to(device)
            i_t = torch.tensor(all_candidates).to(device)
            scores = model(u_t, i_t).cpu().numpy()
            preds = sorted(zip(all_candidates, scores), key=lambda x: x[1], reverse=True)[:100]

            P = calculate_propensity(train_user_interactions.get(user_idx, []), item_to_group, 3, train_user_ratings.get(user_idx, []))

            # Base Metrics
            base_recs = [p[0] for p in preds[:k]]
            n_b, a_b, ap_b = evaluate_recommendation_list(user_idx, base_recs, k, test_df, item_popularity_counts, long_tail_item_indices)
            base_metrics['NDCG'].append(n_b)
            base_metrics['ARP'].append(a_b)
            base_metrics['APLT'].append(ap_b)
            base_metrics['UPD'].append(jensenshannon(P, calculate_propensity(base_recs, item_to_group, 3))**2)

            # Reranked Metrics
            rerank_recs = smooth_xquad_rerank(preds, user_history_items, lambda_val, k)
            n_r, a_r, ap_r = evaluate_recommendation_list(user_idx, rerank_recs, k, test_df, item_popularity_counts, long_tail_item_indices)
            reranked_metrics['NDCG'].append(n_r)
            reranked_metrics['ARP'].append(a_r)
            reranked_metrics['APLT'].append(ap_r)
            reranked_metrics['UPD'].append(jensenshannon(P, calculate_propensity(rerank_recs, item_to_group, 3))**2)

    return {
        "base_model_metrics": {k: np.nanmean(v) for k, v in base_metrics.items()},
        "reranked_model_metrics": {k: np.nanmean(v) for k, v in reranked_metrics.items()}
    }

In [ ]:
from collections import defaultdict

reranking_results = evaluate_xquad_reranking_on_test_set(
    model=mf_model,
    test_df=test_df,
    train_user_item_interactions=train_bpr_dataset.user_item_interactions,
    num_items=num_items,
    item_popularity_counts=item_popularity_counts,
    long_tail_item_indices=long_tail_item_indices,
    k=10,
    lambda_val=0.85
)

print("Base MF Model")
for metric, value in reranking_results["base_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

print()
print("MF Model + xQuAD)")
for metric, value in reranking_results["reranked_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

print()

reranking_results_macr = evaluate_xquad_reranking_on_test_set(
    model=pretrained_macr_mf_model,
    test_df=test_df,
    train_user_item_interactions=train_bpr_dataset.user_item_interactions,
    num_items=num_items,
    item_popularity_counts=item_popularity_counts,
    long_tail_item_indices=long_tail_item_indices,
    k=10,
    lambda_val=0.85
)

print("Base MF + MACR Model")
for metric, value in reranking_results_macr["base_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

print()
print("MF Model + MACR + xQuAD")
for metric, value in reranking_results_macr["reranked_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

Base MF Model
   NDCG: 0.0347
   ARP: 135.2529
   APLT: 0.4451
   UPD: 0.1600

MF Model + xQuAD)
   NDCG: 0.0324
   ARP: 120.6399
   APLT: 0.5184
   UPD: 0.0889

Base MF + MACR Model
   NDCG: 0.0197
   ARP: 84.3113
   APLT: 0.6870
   UPD: 0.2383

MF Model + MACR + xQuAD
   NDCG: 0.0208
   ARP: 89.1785
   APLT: 0.5505
   UPD: 0.0980


In [ ]:
import pandas as pd
import numpy as np
import torch
import math
from scipy.spatial.distance import jensenshannon
from collections import defaultdict


# Helper function to compute user profile distribution P for CP reranking
def user_profile_dist_for_cp(user_idx, train_df, get_group_func):
    user_interactions = train_df[train_df['user_idx'] == user_idx]

    counts = np.zeros(3) # For Head (0), Mid (1), Tail (2)

    for _, row in user_interactions.iterrows():
        group = get_group_func(row['item_idx'])
        # Using binary_rating (1 for positive, 0 for negative, or just count interactions)
        # For simplicity and aligning with the original CP, we can use 1 for any interaction
        counts[group] += 1 # Count of items from each group interacted with

    if counts.sum() == 0:
        return np.ones(3) / 3 # Default uniform distribution if no interactions

    return counts / counts.sum()

# JS Divergence function
def js_divergence(p, q):
    # Add a small epsilon to avoid log(0) if any element is zero
    p = p + 1e-9
    q = q + 1e-9
    m = 0.5 * (p + q)
    return 0.5 * (np.sum(p * np.log2(p / m)) + np.sum(q * np.log2(q / m))) # Original JSD


def cp_rerank(user_idx, candidates_with_scores, train_df, get_group_func, n=10, lambda_=0.5):
    P = user_profile_dist_for_cp(user_idx, train_df, get_group_func)

    selected = []
    selected_groups = []

    # Extract item_idx and prediction score (rel) from candidates
    candidates_only = [item for item, _ in candidates_with_scores]
    scores_map = {item: score for item, score in candidates_with_scores}

    for _ in range(n):
        best_item = None
        best_score = -np.inf

        for item in candidates_only:
            if item in selected:
                continue

            rel = scores_map.get(item, 0) # Get the base model score

            temp_groups = selected_groups + [get_group_func(item)]

            counts = np.zeros(3)
            for g in temp_groups:
                counts[g] += 1

            Q = counts / counts.sum()
            js = js_divergence(P, Q)

            score = (1 - lambda_) * rel - lambda_ * js

            if score > best_score:
                best_score = score
                best_item = item

        if best_item is not None:
            selected.append(best_item)
            selected_groups.append(get_group_func(best_item))
        else:
            break

    return selected



In [ ]:
def evaluate_cp_reranking_on_test_set(model, test_df, train_user_item_interactions, num_items, item_popularity_counts, long_tail_item_indices, k=10, lambda_val=0.5):
    model.eval()
    train_user_interactions_map = train_df.groupby('user_idx')['item_idx'].apply(list).to_dict()
    train_user_ratings_map = train_df.groupby('user_idx')['rating'].apply(list).to_dict()
    get_group_func = build_popularity_groups_for_cp(item_counts_series, num_items)
    results = defaultdict(list)

    with torch.no_grad():
        for user_idx in test_df['user_idx'].unique():
            history = train_user_item_interactions.get(user_idx, set())
            all_c = [i for i in range(num_items) if i not in history]
            if not all_c: continue
            u_t = torch.tensor([user_idx]*len(all_c)).to(device)
            i_t = torch.tensor(all_c).to(device)
            scores = model(u_t, i_t).cpu().numpy()
            preds = sorted(zip(all_c, scores), key=lambda x: x[1], reverse=True)[:100]

            # Use global item_to_group from cell 9K8qr_b5k3YX
            P = calculate_propensity(train_user_interactions_map.get(user_idx, []), item_to_group, 3, train_user_ratings_map.get(user_idx, []))

            # Base Metrics
            base_recs = [p[0] for p in preds[:k]]
            n_b, a_b, ap_b = evaluate_recommendation_list(user_idx, base_recs, k, test_df, item_popularity_counts, long_tail_item_indices)
            results['NDCG'].append(n_b); results['ARP'].append(a_b); results['APLT'].append(ap_b)
            results['UPD'].append(jensenshannon(P, calculate_propensity(base_recs, item_to_group, 3))**2)

            # CP Reranked Metrics
            cp_recs = cp_rerank(user_idx, preds, train_df, get_group_func, k, lambda_val)
            n_r, a_r, ap_r = evaluate_recommendation_list(user_idx, cp_recs, k, test_df, item_popularity_counts, long_tail_item_indices)
            results['NDCG_rerank'].append(n_r); results['ARP_rerank'].append(a_r); results['APLT_rerank'].append(ap_r)
            results['UPD_rerank'].append(jensenshannon(P, calculate_propensity(cp_recs, item_to_group, 3))**2)

    return {
        "base_model_metrics": {k: np.nanmean(v) for k, v in results.items() if '_rerank' not in k},
        "reranked_model_metrics": {k.replace('_rerank', ''): np.nanmean(v) for k, v in results.items() if '_rerank' in k}
    }

In [ ]:
cp_reranking_results_mf = evaluate_cp_reranking_on_test_set(
    model=mf_model,
    test_df=test_df,
    train_user_item_interactions=train_bpr_dataset.user_item_interactions,
    num_items=num_items,
    item_popularity_counts=item_popularity_counts,
    long_tail_item_indices=long_tail_item_indices,
    k=10,
    lambda_val=0.85
)

print("Base MF Model")
for metric, value in cp_reranking_results_mf["base_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

print()

print("MF + CP Reranking")
for metric, value in cp_reranking_results_mf["reranked_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

print()

cp_reranking_results_macr_mf = evaluate_cp_reranking_on_test_set(
    model=pretrained_macr_mf_model,
    test_df=test_df,
    train_user_item_interactions=train_bpr_dataset.user_item_interactions,
    num_items=num_items,
    item_popularity_counts=item_popularity_counts,
    long_tail_item_indices=long_tail_item_indices,
    k=10,
    lambda_val=0.85
)

print("Base MF + MACR Model")
for metric, value in cp_reranking_results_macr_mf["base_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

print()

print("MF + MACR Model + CP Reranking")
for metric, value in cp_reranking_results_macr_mf["reranked_model_metrics"].items():
    print(f"   {metric}: {value:.4f}")

Base MF Model
   NDCG: 0.0347
   ARP: 135.2529
   APLT: 0.4451
   UPD: 0.1600

MF + CP Reranking
   NDCG: 0.0329
   ARP: 122.3584
   APLT: 0.4952
   UPD: 0.0744

Base MF + MACR Model
   NDCG: 0.0197
   ARP: 84.3113
   APLT: 0.6870
   UPD: 0.2383

MF + MACR Model + CP Reranking
   NDCG: 0.0207
   ARP: 85.1007
   APLT: 0.5710
   UPD: 0.0032
